# Choosing a Pattern [Step 08.05 - Four patterns, one task, real numbers]

> **MLCourse - Agentic AI - LangGraph**

Benchmarks in blog posts are usually run on different tasks, with different models,
by people who already prefer one answer. This notebook does the boring, honest thing:

**One task. One model. One session. Same tools. Same grader. Four patterns.**

Every number below is measured live when you run this notebook. Nothing is hard-coded.

### What you'll learn

- A fair measurement harness for agent patterns.
- The real trade curve between **tokens**, **latency** and **success**.
- Why the ranking **changes with the task**, and how to predict which way.
- A decision procedure you can apply to your own work.

### How to read the results

- **Total tokens** is the cost axis (input + output, both billed).
- **Latency** is wall-clock seconds; it tracks LLM *round trips* more than tokens.
- **Correct** comes from a deterministic grader, not from an opinion.

> **Rate limits:** this notebook makes the most model calls in the module. Every
> call goes through `safe_invoke` (paced, with exponential backoff), and there is a
> cool-down between patterns. On Groq's free tier (~8000 TPM) the whole notebook
> fits comfortably; if you re-run it back to back, give it a minute in between.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### The shared task world


In [ ]:
# Every notebook in this module attacks THE SAME task with a different reasoning
# pattern, so the comparison in notebook 05 is apples-to-apples.

from langchain_core.tools import tool

# A tiny deterministic "database". Deterministic matters: we need to check
# correctness automatically, without a human reading the answer.
POPULATION = {"tokyo": 13_960_000, "lagos": 15_400_000, "lima": 9_750_000}
AREA_KM2 = {"tokyo": 2194, "lagos": 1171, "lima": 2672}

TOOL_CALLS = {"count": 0}          # instrumentation: how many tool calls happened


@tool
def population(city: str) -> str:
    """Return the population of a city as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(POPULATION.get(city.strip().lower(), "unknown city"))


@tool
def area_km2(city: str) -> str:
    """Return the land area of a city in square kilometres as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(AREA_KM2.get(city.strip().lower(), "unknown city"))


TOOLS = [population, area_km2]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

TASK = (
    "Among Tokyo, Lagos and Lima, which city has the highest population density "
    "(people per square kilometre)? Answer with the city name and the density "
    "rounded to the nearest whole number."
)

# Ground truth, computed here so the notebook can grade itself.
DENSITIES = {c: POPULATION[c] / AREA_KM2[c] for c in POPULATION}
GT_CITY = max(DENSITIES, key=DENSITIES.get)
GT_DENSITY = round(DENSITIES[GT_CITY])

print("Task:", TASK)
print()
for c in sorted(DENSITIES, key=DENSITIES.get, reverse=True):
    print("  %-6s %9d / %5d = %7.0f people/km2" % (c, POPULATION[c], AREA_KM2[c], DENSITIES[c]))
print()
print("Ground truth -> %s, %d" % (GT_CITY.title(), GT_DENSITY))


def grade(answer: str) -> bool:
    """Automatic grader: the answer must name the right city AND the right density.

    The density is accepted within +/-2 to tolerate rounding differences.
    """
    import re
    if not answer:
        return False
    low = answer.lower()
    if GT_CITY not in low:
        return False
    cleaned = low.replace(",", "").replace(".", " ")
    numbers = [int(n) for n in re.findall(r"\d+", cleaned)]
    return any(abs(n - GT_DENSITY) <= 2 for n in numbers)


### Instrumentation: a token and latency meter


In [ ]:
import time


class Meter:
    """Accumulates token usage, call counts and wall-clock time for one run.

    Every pattern in this module is wrapped in one of these, so notebook 05 can
    compare them on identical instrumentation.
    """

    def __init__(self, name):
        self.name = name
        self.input_tokens = 0
        self.output_tokens = 0
        self.llm_calls = 0
        self.tool_calls = 0
        self.seconds = 0.0
        self._t0 = None

    def start(self):
        TOOL_CALLS["count"] = 0
        self._t0 = time.time()
        return self

    def stop(self):
        self.seconds = time.time() - self._t0
        self.tool_calls = TOOL_CALLS["count"]
        return self

    def record(self, message):
        """Add one AIMessage's usage to the totals, then return the message."""
        usage = getattr(message, "usage_metadata", None) or {}
        if usage:
            self.input_tokens += usage.get("input_tokens", 0)
            self.output_tokens += usage.get("output_tokens", 0)
            self.llm_calls += 1
        return message

    def record_all(self, messages):
        """Add usage from every AIMessage in a list (for create_agent results)."""
        for m in messages:
            if getattr(m, "usage_metadata", None):
                self.record(m)
        return messages

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens

    def report(self, answer=None, correct=None):
        print()
        print("=" * 62)
        print("PATTERN : %s" % self.name)
        print("-" * 62)
        print("LLM calls    : %d" % self.llm_calls)
        print("tool calls   : %d" % self.tool_calls)
        print("input tokens : %d" % self.input_tokens)
        print("output tokens: %d" % self.output_tokens)
        print("TOTAL tokens : %d" % self.total_tokens)
        print("latency      : %.1fs" % self.seconds)
        if correct is not None:
            print("correct      : %s" % ("YES" if correct else "NO"))
        print("=" * 62)
        if answer:
            print(answer)
        return self


### 1. Pattern 1 - ReAct (the baseline)


In [4]:
from typing import Annotated, TypedDict
import operator, re
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent

SYS = ("You are a careful analyst. Use the provided tools to look up facts. "
       "Do the arithmetic yourself. Give a short final answer.")

m_react = Meter("ReAct").start()
_agent = create_agent(model=make_llm(max_tokens=400), tools=TOOLS, system_prompt=SYS)
_out = _agent.invoke({"messages": [("user", TASK)]})
m_react.stop()
m_react.record_all(_out["messages"])
a_react = _out["messages"][-1].content.strip()

m_react.report(a_react.replace("\n", " ")[:220], grade(a_react))
time.sleep(8)          # cool-down between patterns, to stay under the TPM ceiling


PATTERN : ReAct
--------------------------------------------------------------
LLM calls    : 3
tool calls   : 7
input tokens : 1874
output tokens: 309
TOTAL tokens : 2183
latency      : 2.0s
correct      : YES
Tokyo: 13,960,000 / 2,194 ≈ 6,363 Lagos: 15,400,000 / 1,171 ≈ 13,151 Lima: 9,750,000 / 2,672 ≈ 3,649  Lagos has the highest population density.  **Lagos, ≈ 13,151 people/km²**


### 2. Pattern 2 - Reflexion

The actor / evaluator / reflector loop from notebook 02, retargeted at the shared
task. The evaluator is **deterministic** (it uses `grade` plus a diagnosis), which
is what makes Reflexion work at all.

In [5]:
class ReflexState(TypedDict):
    answer: str
    verdict: str
    evidence: str
    reflections: Annotated[list, operator.add]
    attempt: int


REFLEX_MAX = 3
reflex_actor_agent = create_agent(model=make_llm(max_tokens=400), tools=TOOLS,
                                  system_prompt=SYS)


def reflex_actor(state: ReflexState) -> dict:
    n = state.get("attempt", 0) + 1
    prompt = TASK
    if state["reflections"]:
        prompt = ("%s\n\nPrevious attempts failed. Lessons:\n%s\nApply every lesson."
                  % (TASK, "\n".join("- " + r for r in state["reflections"])))
    out = reflex_actor_agent.invoke({"messages": [("user", prompt)]})
    m_reflex.record_all(out["messages"])
    ans = out["messages"][-1].content.strip()
    print("  [actor  ] attempt %d -> %s" % (n, ans.replace("\n", " ")[:70]))
    return {"answer": ans, "attempt": n}


def reflex_eval(state: ReflexState) -> dict:
    """Deterministic evaluator with a specific diagnosis."""
    if grade(state["answer"]):
        return {"verdict": "pass", "evidence": "matches ground truth"}
    if GT_CITY not in state["answer"].lower():
        why = ("named the wrong city; the correct one is whichever has the highest "
               "population divided by area")
    else:
        why = ("named the right city but the density value is wrong; expected about %d"
               % GT_DENSITY)
    return {"verdict": "fail", "evidence": why}


def reflex_reflect(state: ReflexState) -> dict:
    prompt = ("An assistant answered: %s\nAn objective checker said: %s\n"
              "Write ONE short imperative lesson (max 25 words) to prevent this exact "
              "mistake next time. Start with a verb, no preamble."
              % (state["answer"], state["evidence"]))
    lesson = m_reflex.record(safe_invoke(make_llm(max_tokens=120), prompt)).content.strip()
    print("  [reflect] %s" % lesson.replace("\n", " ")[:80])
    return {"reflections": [lesson]}


def reflex_route(state: ReflexState) -> str:
    return "done" if (state["verdict"] == "pass" or state["attempt"] >= REFLEX_MAX) else "reflect"


_rg = StateGraph(ReflexState)
_rg.add_node("actor", reflex_actor)
_rg.add_node("evaluate", reflex_eval)
_rg.add_node("reflect", reflex_reflect)
_rg.add_edge(START, "actor")
_rg.add_edge("actor", "evaluate")
_rg.add_conditional_edges("evaluate", reflex_route, {"reflect": "reflect", "done": END})
_rg.add_edge("reflect", "actor")
reflexion_app = _rg.compile()

m_reflex = Meter("Reflexion").start()
out_reflex = reflexion_app.invoke({"answer": "", "verdict": "", "evidence": "",
                                   "reflections": [], "attempt": 0})
m_reflex.stop()
a_reflex = out_reflex["answer"]
m_reflex.report(a_reflex.replace("\n", " ")[:220], grade(a_reflex))
print("attempts:", out_reflex["attempt"], "| lessons learned:", len(out_reflex["reflections"]))
time.sleep(8)

  [actor  ] attempt 1 -> Tokyo: 13,960,000 / 2,194 ≈ 6,363 Lagos: 15,400,000 / 1,171 ≈ 13,151 L

PATTERN : Reflexion
--------------------------------------------------------------
LLM calls    : 3
tool calls   : 7
input tokens : 1874
output tokens: 309
TOTAL tokens : 2183
latency      : 2.1s
correct      : YES
Tokyo: 13,960,000 / 2,194 ≈ 6,363 Lagos: 15,400,000 / 1,171 ≈ 13,151 Lima: 9,750,000 / 2,672 ≈ 3,649  Lagos has the highest population density.  **Lagos, ≈ 13,151 people/km²**
attempts: 1 | lessons learned: 0


### 3. Pattern 3 - Plan-and-Execute


In [6]:
class PEState(TypedDict):
    plan: list
    past_steps: Annotated[list, operator.add]
    response: str
    cycles: int


PE_MAX_CYCLES = 5
pe_exec_agent = create_agent(model=make_llm(max_tokens=320), tools=TOOLS,
                             system_prompt="Execute ONE step. Use tools for facts. Be brief.")


def pe_plan(state: PEState) -> dict:
    prompt = ("You are a planner. Break the objective into a minimal numbered list of "
              "steps. Each step must be executable alone with the tools "
              "population(city) and area_km2(city). The last step must state the answer. "
              "Output ONLY the numbered steps.\n\nObjective: " + TASK)
    raw = m_pe.record(safe_invoke(make_llm(max_tokens=300), prompt)).content
    steps = [re.sub(r"^\s*\d+[\.\)]?\s*", "", ln).strip()
             for ln in raw.splitlines() if ln.strip()]
    steps = [s for s in steps if len(s) > 5][:6]
    print("  [planner] %d steps" % len(steps))
    return {"plan": steps, "cycles": 0}


def pe_exec(state: PEState) -> dict:
    step = state["plan"][0]
    hist = ""
    if state["past_steps"]:
        hist = "\nDone so far:\n" + "\n".join("- %s -> %s" % (s, r)
                                              for s, r in state["past_steps"][-3:])
    out = pe_exec_agent.invoke({"messages": [
        ("user", "Objective: %s%s\n\nExecute ONLY: %s" % (TASK, hist, step))]})
    m_pe.record_all(out["messages"])
    res = out["messages"][-1].content.strip().replace("\n", " ")
    print("  [exec   ] %s -> %s" % (step[:42], res[:52]))
    return {"past_steps": [(step, res)], "plan": state["plan"][1:]}


def pe_replan(state: PEState) -> dict:
    done = "\n".join("- %s -> %s" % (s, r) for s, r in state["past_steps"])
    left = "\n".join("- " + s for s in state["plan"]) or "(none)"
    prompt = ("Objective: %s\n\nCompleted:\n%s\n\nRemaining:\n%s\n\n"
              "If the completed work answers the objective, reply exactly:\n"
              "ANSWER: <final answer>\nOtherwise reply with the revised remaining "
              "steps as a numbered list only." % (TASK, done, left))
    raw = m_pe.record(safe_invoke(make_llm(max_tokens=300), prompt)).content.strip()
    cycles = state.get("cycles", 0) + 1
    if raw.upper().startswith("ANSWER:"):
        print("  [replan ] FINISH")
        return {"response": raw.split(":", 1)[1].strip(), "plan": [], "cycles": cycles}
    steps = [re.sub(r"^\s*\d+[\.\)]?\s*", "", ln).strip()
             for ln in raw.splitlines() if ln.strip()]
    steps = [s for s in steps if len(s) > 5][:5]
    print("  [replan ] %d step(s) left" % len(steps))
    return {"plan": steps, "cycles": cycles}


def pe_route(state: PEState) -> str:
    if state["response"]:
        return "done"
    if not state["plan"] or state["cycles"] >= PE_MAX_CYCLES:
        return "force"
    return "execute"


def pe_force(state: PEState) -> dict:
    done = "\n".join("- %s -> %s" % (s, r) for s, r in state["past_steps"])
    prompt = ("Objective: %s\n\nEvidence:\n%s\n\nState the final answer in one sentence."
              % (TASK, done))
    return {"response": m_pe.record(safe_invoke(make_llm(max_tokens=200), prompt)).content.strip()}


_pg = StateGraph(PEState)
_pg.add_node("planner", pe_plan)
_pg.add_node("executor", pe_exec)
_pg.add_node("replanner", pe_replan)
_pg.add_node("force", pe_force)
_pg.add_edge(START, "planner")
_pg.add_edge("planner", "executor")
_pg.add_edge("executor", "replanner")
_pg.add_conditional_edges("replanner", pe_route,
                          {"execute": "executor", "force": "force", "done": END})
_pg.add_edge("force", END)
plan_exec_app = _pg.compile()

m_pe = Meter("Plan-and-Execute").start()
out_pe = plan_exec_app.invoke({"plan": [], "past_steps": [], "response": "", "cycles": 0})
m_pe.stop()
a_pe = out_pe["response"]
m_pe.report(a_pe.replace("\n", " ")[:220], grade(a_pe))
print("steps executed:", len(out_pe["past_steps"]), "| replan cycles:", out_pe["cycles"])
time.sleep(8)

  [planner] 4 steps


  [exec   ] Calculate the population density for Tokyo -> Tokyo density = 13,960,000 ÷ 2,194 ≈ **6,363 people/


  [replan ] 3 step(s) left


  [exec   ] Calculate the population density for Lagos -> Lagos density = 15,400,000 ÷ 1,171 ≈ **13,151 people


  [replan ] 3 step(s) left


  [exec   ] Calculate the population density for Lima  -> Lima density = 9,750,000 ÷ 2,672 ≈ **3,649 people/km


  [replan ] FINISH

PATTERN : Plan-and-Execute
--------------------------------------------------------------
LLM calls    : 10
tool calls   : 6
input tokens : 4351
output tokens: 571
TOTAL tokens : 4922
latency      : 13.7s
correct      : YES
Lagos, 13,151
steps executed: 3 | replan cycles: 3


### 4. Pattern 4 - ReWOO


In [7]:
PLAN_LINE = re.compile(r"#(E\d+)\s*=\s*(\w+)\s*\[([^\]]*)\]")

REWOO_PROMPT = """You write executable plans. Each step is a tool call stored in a
variable #E1, #E2, ...

Available tools:
  population[city]  -- population of the city as a number
  area_km2[city]    -- land area of the city in square kilometres

Format each step on exactly two lines:
Plan: <one short sentence>
#E<n> = <tool>[<argument>]

Do NOT do arithmetic in the plan - only tool calls. Output the plan and nothing else.

Example for "What is the population of Paris and of Rome?":
Plan: Look up the population of Paris.
#E1 = population[Paris]
Plan: Look up the population of Rome.
#E2 = population[Rome]

Task: {task}
"""

m_rewoo = Meter("ReWOO").start()

# --- LLM call 1 of 2: plan -----------------------------------------------------
raw_plan = m_rewoo.record(safe_invoke(make_llm(max_tokens=520),
                                      REWOO_PROMPT.format(task=TASK))).content
rewoo_steps = []
for line in raw_plan.splitlines():
    hit = PLAN_LINE.search(line.strip())
    if hit and hit.group(2) in TOOLS_BY_NAME:
        rewoo_steps.append(("#" + hit.group(1), hit.group(2), hit.group(3).strip()))
print("  [planner] %d executable steps" % len(rewoo_steps))

# --- Worker: ZERO LLM calls ----------------------------------------------------
rewoo_results = {}
for var, tool_name, arg in rewoo_steps:
    resolved = arg
    for pv, val in rewoo_results.items():
        resolved = resolved.replace(pv, str(val))
    rewoo_results[var] = TOOLS_BY_NAME[tool_name].invoke({"city": resolved})
print("  [worker ] executed %d tool calls with no model" % len(rewoo_results))

# --- LLM call 2 of 2: solve ----------------------------------------------------
# Every row is LABELLED with the tool and argument that produced it. A bare
# "#E1 = 13960000" would leave the solver guessing which city it refers to.
evidence = "\n".join("%s = %s[%s] -> %s" % (v, t, a, rewoo_results[v])
                     for v, t, a in rewoo_steps if v in rewoo_results)
solve_prompt = ("Task: %s\n\nEvidence from an executed plan:\n%s\n\nPlan:\n%s\n\n"
                "Using ONLY the evidence, do any arithmetic needed and give the final "
                "answer in one short sentence." % (TASK, evidence, raw_plan))
a_rewoo = m_rewoo.record(safe_invoke(make_llm(max_tokens=340), solve_prompt)).content.strip()
m_rewoo.stop()

m_rewoo.report(a_rewoo.replace("\n", " ")[:220], grade(a_rewoo))

  [planner] 6 executable steps
  [worker ] executed 6 tool calls with no model



PATTERN : ReWOO
--------------------------------------------------------------
LLM calls    : 2
tool calls   : 6
input tokens : 526
output tokens: 144
TOTAL tokens : 670
latency      : 4.4s
correct      : YES
Lagos has the highest population density at 13,151 people per square kilometre.


### 5. The comparison table

All four, measured in this session.

In [8]:
METERS = [m_react, m_reflex, m_pe, m_rewoo]
ANSWERS = {m_react.name: a_react, m_reflex.name: a_reflex,
           m_pe.name: a_pe, m_rewoo.name: a_rewoo}

print("%-18s %5s %5s %8s %8s %8s %7s %8s"
      % ("pattern", "LLM", "tool", "in tok", "out tok", "TOTAL", "sec", "correct"))
print("-" * 76)
for m in METERS:
    print("%-18s %5d %5d %8d %8d %8d %7.1f %8s"
          % (m.name, m.llm_calls, m.tool_calls, m.input_tokens, m.output_tokens,
             m.total_tokens, m.seconds, "YES" if grade(ANSWERS[m.name]) else "NO"))

pattern              LLM  tool   in tok  out tok    TOTAL     sec  correct
----------------------------------------------------------------------------
ReAct                  3     7     1874      309     2183     2.0      YES
Reflexion              3     7     1874      309     2183     2.1      YES
Plan-and-Execute      10     6     4351      571     4922    13.7      YES
ReWOO                  2     6      526      144      670     4.4      YES


In [9]:
cheapest = min(METERS, key=lambda m: m.total_tokens)
fastest = min(METERS, key=lambda m: m.seconds)
correct_ones = [m.name for m in METERS if grade(ANSWERS[m.name])]

print("cheapest (tokens) :", cheapest.name, "at %d tokens" % cheapest.total_tokens)
print("fastest (latency) :", fastest.name, "at %.1fs" % fastest.seconds)
print("correct           :", ", ".join(correct_ones) if correct_ones else "none")
print()
print("relative token cost (cheapest = 1.00x)")
for m in sorted(METERS, key=lambda m: m.total_tokens):
    print("  %-18s %5.2fx  (%d tokens)"
          % (m.name, m.total_tokens / cheapest.total_tokens, m.total_tokens))
print()
print("relative latency (fastest = 1.00x)")
for m in sorted(METERS, key=lambda m: m.seconds):
    print("  %-18s %5.2fx  (%.1fs)" % (m.name, m.seconds / fastest.seconds, m.seconds))

cheapest (tokens) : ReWOO at 670 tokens
fastest (latency) : ReAct at 2.0s
correct           : ReAct, Reflexion, Plan-and-Execute, ReWOO

relative token cost (cheapest = 1.00x)
  ReWOO               1.00x  (670 tokens)
  ReAct               3.26x  (2183 tokens)
  Reflexion           3.26x  (2183 tokens)
  Plan-and-Execute    7.35x  (4922 tokens)

relative latency (fastest = 1.00x)
  ReAct               1.00x  (2.0s)
  Reflexion           1.04x  (2.1s)
  ReWOO               2.19x  (4.4s)
  Plan-and-Execute    6.86x  (13.7s)


### 6. Reading the results honestly

Four things must be said explicitly, because a single benchmark run invites
over-reading.

**1. This task is small.** Six independent lookups and one division. That is exactly
the regime where ReWOO looks best and Plan-and-Execute looks worst, because the
planner and replanner overhead is amortised over almost nothing. On a 30-step task
the ranking between those two moves.

**2. Reflexion's cost depends entirely on whether attempt 1 succeeds.** If the actor
gets it right first time, Reflexion costs the same as ReAct plus a free deterministic
check. If it fails twice, it costs roughly three times ReAct. High variance is its
defining property, and this run shows only one draw from that distribution.

**3. Latency tracks round trips, not tokens.** A pattern with fewer, larger LLM calls
beats one with many small calls, because network round trips dominate. Note the
pacing sleeps inside `safe_invoke` are included in these timings - they are part of
living within a rate limit.

**4. This is n=1.** These are single runs of a stochastic system. For a real decision
you would run each pattern 20+ times across 20+ tasks and compare distributions. The
harness above is exactly what you would loop.

Let's make point 1 concrete by projecting each pattern's cost as the task grows.

In [10]:
print("Cost model per pattern, as a function of N tool-relevant steps:")
print()
print("  ReAct             ~ O(N^2) input tokens, N+1 LLM calls")
print("  Reflexion         ~ (attempts) x ReAct  +  (attempts-1) reflection calls")
print("  Plan-and-Execute  ~ O(N) LLM calls (1 planner + N executors + N replanners)")
print("  ReWOO             ~ O(1) LLM calls (exactly 2); evidence table grows O(N)")
print()

base = m_react.input_tokens / max(1, m_react.llm_calls)
attempts = max(1, out_reflex["attempt"])

print("%6s %12s %12s %14s %10s" % ("N", "ReAct", "Reflexion", "Plan+Execute", "ReWOO"))
print("-" * 60)
for n in (3, 6, 12, 25, 50):
    react_p = sum(base * 0.6 + base * 0.4 * k for k in range(n))
    reflex_p = react_p * attempts
    pe_p = n * (base * 0.9) + n * (base * 0.5)     # executor + replanner per step
    rewoo_p = m_rewoo.input_tokens + 8 * n
    print("%6d %12d %12d %14d %10d" % (n, react_p, reflex_p, pe_p, rewoo_p))
print()
print("(Projections extrapolate THIS run's measured per-call sizes. They show the")
print(" SHAPE of each cost curve, not a promise about your workload.)")

Cost model per pattern, as a function of N tool-relevant steps:

  ReAct             ~ O(N^2) input tokens, N+1 LLM calls
  Reflexion         ~ (attempts) x ReAct  +  (attempts-1) reflection calls
  Plan-and-Execute  ~ O(N) LLM calls (1 planner + N executors + N replanners)
  ReWOO             ~ O(1) LLM calls (exactly 2); evidence table grows O(N)

     N        ReAct    Reflexion   Plan+Execute      ReWOO
------------------------------------------------------------
     3         1874         1874           2623        550
     6         5996         5996           5247        574
    12        20988        20988          10494        622
    25        84330        84330          21863        726
    50       324826       324826          43726        926

(Projections extrapolate THIS run's measured per-call sizes. They show the
 SHAPE of each cost curve, not a promise about your workload.)


### 7. The decision procedure

Work down this list and stop at the first match.

**1. Is the task one or two tool calls, or exploratory with an unknown shape?**
-> **ReAct.** Do not add machinery to a problem that does not have it. ReAct is the
right default and most tasks never outgrow it.

**2. Can you verify the answer cheaply and objectively** (tests, a compiler, a schema,
a ground truth)? -> **Reflexion**, wrapped around whichever base pattern you use.
Grounded verification is what makes retries productive rather than random.

**3. Is the work long, and does someone need to approve or audit the approach before
it runs?** -> **Plan-and-Execute.** The plan is a reviewable artifact, and the
executor can be a cheaper model than the planner.

**4. Is the work tool-heavy with statically knowable calls, and is cost or latency the
binding constraint?** -> **ReWOO.** Two LLM calls regardless of tool count.

**5. Does step *k* depend on *interpreting* step *k-1*'s result?**
-> **Not ReWOO.** Use ReAct or Plan-and-Execute.

### They compose

These are layers, not rivals:

- **Reflexion around ReWOO** - cheap execution, retried with a lesson when the solver
  reports insufficient evidence.
- **Plan-and-Execute with ReAct executors** - a global plan, each step handled by a
  small adaptive agent.
- **ReWOO first, ReAct fallback** - pay ReWOO's price on the easy majority, escalate
  the hard tail.

Every one of those is a **subgraph composition**, which is exactly what
[07_subgraphs_and_composition](../07_subgraphs_and_composition/README.md) taught you
to build.

In [11]:
print("SUMMARY OF THIS RUN")
print("=" * 78)
for m in METERS:
    print("%-18s tokens=%-6d llm_calls=%-3d latency=%-6.1fs correct=%s"
          % (m.name, m.total_tokens, m.llm_calls, m.seconds,
             "YES" if grade(ANSWERS[m.name]) else "NO"))
print("=" * 78)
print()
print("answers given:")
for name, ans in ANSWERS.items():
    print("  %-18s %s" % (name, ans.replace("\n", " ")[:92]))
print()
print("ground truth: %s, %d people/km2" % (GT_CITY.title(), GT_DENSITY))

SUMMARY OF THIS RUN
ReAct              tokens=2183   llm_calls=3   latency=2.0   s correct=YES
Reflexion          tokens=2183   llm_calls=3   latency=2.1   s correct=YES
Plan-and-Execute   tokens=4922   llm_calls=10  latency=13.7  s correct=YES
ReWOO              tokens=670    llm_calls=2   latency=4.4   s correct=YES

answers given:
  ReAct              Tokyo: 13,960,000 / 2,194 ≈ 6,363 Lagos: 15,400,000 / 1,171 ≈ 13,151 Lima: 9,750,000 / 2,672
  Reflexion          Tokyo: 13,960,000 / 2,194 ≈ 6,363 Lagos: 15,400,000 / 1,171 ≈ 13,151 Lima: 9,750,000 / 2,672
  Plan-and-Execute   Lagos, 13,151
  ReWOO              Lagos has the highest population density at 13,151 people per square kilometre.

ground truth: Lagos, 13151 people/km2


### 8. Reuse this harness

The `Meter` class plus a deterministic `grade` function is the whole evaluation
harness, and it is the transferable part of this notebook. To evaluate patterns on
*your* problem:

1. Replace `TASK`, the tools, and `grade` with your own.
2. Keep the meters identical across patterns - that is what makes it a comparison
   rather than four anecdotes.
3. Loop each pattern 20 times and report the median plus the success **rate**, not a
   single run.
4. Add a token **budget** per pattern and count budget exhaustion as a failure.

### Recap

- Measured on one shared task with one model, the four patterns differ substantially
  in tokens, LLM calls and latency - the table above is the evidence.
- **ReAct**: cheapest to build, quadratic in context, naturally adaptive.
- **Reflexion**: highest variance; pays off only with a grounded evaluator.
- **Plan-and-Execute**: reviewable and cheap to execute, but pays planner overhead.
- **ReWOO**: cheapest by far on tool-heavy work, and rigid.
- The right answer is task-dependent, and the patterns compose.

### Where to go next

**[09_travel_planner](../09_travel_planner/README.md)** - the capstone, where these
reasoning patterns and the composition techniques from module 07 come together in a
full application.